In [0]:
import pyspark.pandas as pd
from pyspark.sql.functions import year, month, weekofyear

In [0]:
# load necessary tables 
activiteiten = spark.table("bronze.vcs_bi_zis_tijdschrijven_zpm").filter("soort_ggz_zorg = 'ZVW' AND declarabel = 'Ja'")
ozp = spark.table("bronze.vcs_bi_zis_ozp").filter("soort_ggz_zorg = 'ZVW' AND declarabel = 'Ja'")
verblijf = spark.table("bronze.vcs_bi_zis_verblijf").filter("soort_ggz_zorg = 'ZVW' AND declarabel = 'Ja'")
diensten = spark.table("bronze.vcs_bi_personeel_dienst")
productiviteit = spark.table("bronze.vcs_bi_personeel_productiviteit")

In [0]:
# filter necessary columns
activiteiten_gefilterd = activiteiten.select(
    "contactnummer", 
    "contactcode", 
    "contactomschrijving", 
    "rapportagedatum", 
    "begintijd", 
    "eindtijd", 
    "medewerker", 
    "beroepscategorie_code",
    "beroepscategorie",
    "beroepcode",
    "medewerkerberoep", 
    "org_level_1", 
    "org_level_2", 
    "org_level_3", 
    "org_level_4", 
    "tijd_client_direct_minuten", 
    "tijd_client_indirect_minuten", 
    "tijd_client_reis_minuten", 
    "tijd_client_minuten", 
    "tijd_medewerker_direct_minuten", 
    "tijd_medewerker_indirect_minuten", 
    "declaratiecode", 
    "declaratiecode_omschrijving", 
    "declaratiecode_categorie", 
    "groepsgrootte", 
    "bedrag_totaal", 
    "bedrag_totaal_nza", 
    "setting_code", 
    "setting", 
    "consult_type", 
    "type_financier", 
    "financier", 
    "client_code", 
    "no_show", 
    "declarabel", 
    "zorgtraject", 
    "diagnose", 
    "diagnosecode", 
    "diagnosegroep", 
    "diagnosegroepcode", 
    "meer_dan_365_dagen_zvw_verblijf", 
    "cruciale_zorg", 
    "zorgprogramma_omschrijving", 
    "zorgvraagtypering"
    )

In [0]:
# filter necessary columns
ozp_gefilterd = ozp.select(
    "contactnummer", 
    "contactcode", 
    "contactomschrijving", 
    "rapportagedatum", 
    "begintijd", 
    "eindtijd", 
    "medewerker", 
    "beroepscategorie_code",
    "beroepscategorie",
    "beroepcode",
    "medewerkerberoep", 
    "org_level_1", 
    "org_level_2", 
    "org_level_3", 
    "org_level_4", 
    "tijd_client_direct_minuten", 
    "tijd_client_indirect_minuten", 
    "tijd_client_reis_minuten", 
    "tijd_client_minuten", 
    "tijd_medewerker_direct_minuten", 
    "tijd_medewerker_indirect_minuten", 
    "declaratiecode", 
    "declaratiecode_omschrijving", 
    "declaratiecode_categorie", 
    "bedrag_totaal", 
    "bedrag_totaal_nza", 
    "consult_type", 
    "type_financier", 
    "financier", 
    "client_code", 
    "no_show", 
    "crisis_binnen_budget", 
    "declarabel", 
    "meer_dan_365_dagen_zvw_verblijf", 
    "cruciale_zorg", 
    "zorgvraagtypering", 
    "zorgtraject"
    )

In [0]:
# filter necessary columns
verblijf_gefilterd = verblijf.select(
    "verblijfsdagnummer", 
    "zorgtraject", 
    "rapportagedatum", 
    "declaratiecode", 
    "declaratiecode_omschrijving", 
    "declaratiecode_categorie", 
    "financier", 
    "type_financier", 
    "declarabel", 
    "org_level_1", 
    "org_level_2", 
    "org_level_3", 
    "org_level_4", 
    "client_code", 
    "verblijf_op_separeer", 
    "beveiligde_setting", 
    "cruciale_zorg", 
    "meer_dan_365_dagen_zvw_verblijf", 
    "zorgvraagtypering", 
    "type_zvw_zorg", 
    "bedrag_totaal", 
    "bedrag_totaal_nza"
    )

In [0]:
# combine all activities to one dataset
df_declaraties = (activiteiten_gefilterd
      .unionByName(ozp_gefilterd, allowMissingColumns=True)
      .unionByName(verblijf_gefilterd, allowMissingColumns=True))

df_declaraties = (df_declaraties
.withColumn("rapportagejaar", year("rapportagedatum"))
.withColumn("rapportagemaand", month("rapportagedatum"))
.withColumn("rapportageweek", weekofyear("rapportagedatum"))
)

display(df_declaraties.limit(10))

In [0]:
# filter necessary columns for productivitity
df_productiviteit = productiviteit.select(
    "rapportagedatum", 
    "medewerker", 
    "personeelsnummer", 
    "geslacht", 
    "dienstverband_begindatum", 
    "vast_dienstverband", 
    "functie", 
    "functie_groep", 
    "beroepscategorie_code",
    "beroepscategorie",
    "leeftijd", 
    "leeftijdscategorie_per_5_jaar", 
    "leeftijdscategorie_per_10_jaar", 
    "in_loondienst", 
    "org_level_1", 
    "org_level_2", 
    "org_level_3", 
    "org_level_4", 
    "behandelfractie", 
    "contracturen_bruto", 
    "ziek_uren_excl_zwangerschap", 
    "zwangerschapsverlof_uren", 
    "verlof_uren_ouderschap", 
    "verlof_uren_ouderschap_betaald", 
    "verlof_uren_geboorte", 
    "verlof_uren_geboorte_aanvullend", 
    "verlof_uren_onbetaald", 
    "verlof_uren_zorg", 
    "verlof_uren_pleegzorg", 
    "verlof_uren_adoptie", 
    "verlof_uren_excl_ouderschap_of_onbetaald", 
    "bijzonder_verlof", 
    "opleiding_uren_gecorrigeerd", 
    "or_uren_gecorrigeerd", 
    "overige_neventaken_uren_gecorrigeerd", 
    "verlofsparen_uren", 
    "verlofrecht_uren", 
    "productieve_uren", 
    "declarabele_uren", 
    "declarabele_uren_totaal", 
    "declarabele_uren_direct", 
    "declarabele_uren_indirect",  
    "declarabele_uren_indirect_norm_zpm", 
    "declarabele_uren_reistijd", 
    "declarabele_uren_overig", 
    "contracturen_netto", 
    "contracturen_netto_behandelfractie", 
    "roosteruren_netto_behandelfractie", 
    "uren_bruto", "uren_netto", 
    "niet_inzetbare_uren", 
    "aantal_uren_per_fte"
    )

In [0]:
display(diensten.limit(10))

In [0]:
# filter necessary columns for worked hours
df_diensten = diensten.select(
    "dienst_uren", 
    "dienst_soort", 
    "dienst_groep", 
    "flexpool_inzet", 
    "werkzame_uren", 
    "betaalbare_uren", 
    "directe_uren", 
    "dienst_hoofdsoort", 
    "org_level_1", 
    "org_level_2", 
    "org_level_3", 
    "org_level_4", 
    "medewerker", 
    "personeelsnummer", 
    "rapportagedatum", 
)

In [0]:
# Show shape of the dataframes
print("Shape of df_declaraties = " + str((df_declaraties.count(), len(df_declaraties.columns))))
print("Shape of df_productiviteit = " + str((df_productiviteit.count(), len(df_productiviteit.columns))))
print("Shape of df_diensten = " + str((df_diensten.count(), len(df_diensten.columns))))

In [0]:
# Show dataypes of the dataframes
df_declaraties.printSchema()

In [0]:
# Show dataypes of the dataframes
df_productiviteit.printSchema()

In [0]:
# Show dataypes of the dataframes
df_diensten.printSchema()

In [0]:
# Create pandas dataframes 
pd_df_declaraties = df_declaraties.pandas_api()
pd_df_productiviteit = df_productiviteit.pandas_api()
pd_df_diensten = df_diensten.pandas_api()

In [0]:
# Missing values
print("Missing values in df_declaraties:", pd_df_declaraties.isnull().sum())

# Further research for fillna()
#lege_declaraties = pd_df_declaraties[pd_df_declaraties["declaratiecode"].isnull()]
#lege_declaraties["declarabel"].unique()
# contains non-declarable and declarable hours, so further research for fillna() is necessary

#lege_contactnummers = pd_df_declaraties[pd_df_declaraties["contactnummer"].isnull()]
#lege_contactnummers["declaratiecode_categorie"].unique()
# contains none declaratiecode_categorie or declaratiecode_categorie = Verblijf (bed occupancy), which has no contactnumber, so fillna(0) is sufficient

lege_beroepcategorie = pd_df_declaraties[pd_df_declaraties["beroepscategorie"].isnull()]
lege_beroepcategorie["medewerkerberoep"].unique()
# contains non-declarable and declarable hours, beroepscategorie can be filled based on the "functie"

# other columns can be filled using 0 or "Nee"


In [0]:
# Missing values
print("Missing values in df_productiviteit:", pd_df_productiviteit.isnull().sum())
lege_beroepcategorie["declarabel"].unique()
# beroepscategorie can be filled using the "functie"
# Other values fillna(0)

In [0]:
# Missing values
print("Missing values in df_diensten:", pd_df_diensten.isnull().sum())
# no missing values

In [0]:
pd_df_declaraties.describe()

In [0]:
pd_df_productiviteit.describe()

In [0]:
pd_df_diensten.describe()
# lots of columns only containing NaN, could be skipped

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

# sample df
pd_df_sample = pd_df_declaraties.sample(frac=0.05, random_state=42)

# sample df to real df instead of pandas_api
df_sample = pd_df_sample.to_pandas()

# plt figure
plt.figure(figsize=(10, 6))
sns.histplot(data=df_sample, x="bedrag_totaal", kde=True)
plt.show()

In [0]:
# use sample for further analysis
omzet_beroepscategorie = df_sample.groupby("beroepscategorie")["bedrag_totaal"].sum().sort_values(ascending=False)
uc_cluster = df_sample.groupby("org_level_1")["client_code"].nunique()
uc_financier = df_sample.groupby("financier")["client_code"].nunique()
omzet_financier = df_sample.groupby("financier")["bedrag_totaal"].sum()
omzet_per_client = omzet_financier/uc_financier

In [0]:
# focus on specific clients with ECT of rTMS treatments
df_zk_ECT = df_sample[(df_sample["financier"] == "ZILVERENKRUIS") & (df_sample["org_level_3"] == "8915 - OUP Pr. ECT")]
df_zk_ECT["bedrag_totaal"].sum()

In [0]:
pd_df_declaraties.head(5)

In [0]:
# aansluiting met ValueCare - omzet 

pd_df_declaraties[
    (pd_df_declaraties["rapportagejaar"] == 2025)&
    (pd_df_declaraties["crisis_binnen_budget"] != "Ja")
    ].groupby("declaratiecode_categorie")["bedrag_totaal"].sum()

In [0]:
# aansluiting met ValueCare - unieke clienten

pd_df_declaraties[
    (pd_df_declaraties["rapportagejaar"] == 2025)&
    (pd_df_declaraties["crisis_binnen_budget"] != "Ja")
    ]["client_code"].nunique()

In [0]:
# aansluiting met ValueCare - directe uren

pd_df_declaraties[
    (pd_df_declaraties["rapportagejaar"] == 2025)&
    (pd_df_declaraties["crisis_binnen_budget"] != "Ja")
    ].groupby("declaratiecode_categorie")["tijd_client_minuten"].sum()/60